In [73]:
import tskit 
import pandas as pd
import numpy as np
from lets_plot import *
LetsPlot.setup_html()
import pickle
import numpy as np
import seaborn as sns
import cloudpickle
chromosome = 16

"""
coaltime_NoSamples = ts_meta.coalescence_time_distribution(weight_func="pair_coalescence_events")

# %%
#time_windows = coaltime_NoSamples.quantile(np.append(0, 10 ** np.linspace(2, 6, (6-2)*25))/ 10**6)
timeBreaks = np.append(0, 10 ** np.linspace(2, 6, (6-2)*25))
#time_windows = coaltime_NoSamples.quantile()
coal_rate = coaltime.coalescence_rate_in_intervals(timeBreaks)

# %%
output = open('CoalRates_Countries_' + str(chromosome) + '.pkl', 'wb')
pickle.dump(coal_rate, output)
output.close()
"""

'\ncoaltime_NoSamples = ts_meta.coalescence_time_distribution(weight_func="pair_coalescence_events")\n\n# %%\n#time_windows = coaltime_NoSamples.quantile(np.append(0, 10 ** np.linspace(2, 6, (6-2)*25))/ 10**6)\ntimeBreaks = np.append(0, 10 ** np.linspace(2, 6, (6-2)*25))\n#time_windows = coaltime_NoSamples.quantile()\ncoal_rate = coaltime.coalescence_rate_in_intervals(timeBreaks)\n\n# %%\noutput = open(\'CoalRates_Countries_\' + str(chromosome) + \'.pkl\', \'wb\')\npickle.dump(coal_rate, output)\noutput.close()\n'

In [74]:
def lineages_colours():
    return {
        "A": "#feb72d", 
        "O": "#a8db34",
        "M": "#228c8d", 
        "C": "#802694"
    }


def subspecies_colours():
    return {
        "Capensis": "#f7e225",
        "Scutellata": "#feb72d",
        "Unicolor": "#f07f4f",
        "Ligustica": "#310597",
        "Carnica": "#ad2793",
        "Mellifera": "#228c8d",
        "Iberiensis": "#87e5e6",
        "Caucasia": "#a8db34",
        "Buckfast": "#9298a1"
    }

def get_subsp_colour(subsp_list):
    return [subspecies_colours()[x] for x in subsp_list]


def get_lineage_colour(lineage_list):
    return [lineages_colours()[x] for x in lineage_list]


In [75]:
timeBreaks = np.append(0, 10 ** np.linspace(2, 6, (6-2)*10))
#timeBreaks = np.concatenate((np.linspace(0, 10_000, 10), np.linspace(10_000, 100_000, 10), np.linspace(100_000, 1_000_000, 10)))

In [76]:
coaltimeraw_l = cloudpickle.load(open("CoalRateRaw_Lineages_" + str(chromosome) + ".pkl", "rb"))
coal_rate_l = coaltimeraw_l.coalescence_rate_in_intervals(timeBreaks)

coaltimeraw_s = cloudpickle.load(open("CoalRateRaw_Subspecies_" + str(chromosome) + ".pkl", "rb"))
coal_rate_s = coaltimeraw_s.coalescence_rate_in_intervals(timeBreaks)

coaltimeraw_c = cloudpickle.load(open("CoalRateRaw_Country_" + str(chromosome) + ".pkl", "rb"))
coal_rate_c = coaltimeraw_c.coalescence_rate_in_intervals(timeBreaks)

In [77]:

countries = pd.read_csv("Chr" + str(chromosome) + "_CountriesNames.csv").Subspecies.to_list()
subspecies = pd.read_csv("Chr" + str(chromosome) + "_SubspeciesNames.csv").Subspecies.to_list()
lineages = pd.read_csv("Chr" + str(chromosome) + "_LineagesNames.csv").Subspecies.to_list()

In [78]:
lineagesPairs = []
for num1, name1 in enumerate(lineages):
    for name2 in lineages[num1:]:
        lineagesPairs.append((name1, name2))

subspeciesPairs = []
for num1, name1 in enumerate(subspecies):
    for name2 in subspecies[num1:]:
        subspeciesPairs.append((name1, name2))

In [79]:
coalRates_flatten_l = []
for pair in range(len(lineagesPairs)):
    coalRates_flatten_l.append(coal_rate_l[pair, :, :].flatten())



In [80]:
coalRateDF_l = pd.DataFrame.from_records(coalRates_flatten_l).T
coalRateDF_l.columns = lineagesPairs
coalRateDF_l.loc[:, "Time"] = timeBreaks[1:]
coalRateDFL_l = pd.melt(coalRateDF_l, id_vars = "Time", var_name = "LineagesPair", value_name="CoalRate")

coalRateDFL_l.loc[:, "PairType"] = ["Within" if x[0] == x[1] else "Cross" for x in coalRateDFL_l.LineagesPair]
coalRateDFL_l.loc[:, "Ne"] = 1 / coalRateDFL_l.CoalRate
coalRateDFL_l.loc[:, "LogNe"] = np.log10(coalRateDFL_l.Ne)
coalRateDFL_l.loc[:, "LogTime"] = np.log10(coalRateDFL_l.Time)
coalRateDFL_l.loc[:, "LogCoalRate"] = np.log10(coalRateDFL_l.CoalRate)

coalRateDFL_l.loc[:, "Lineage1"] = [x[0] for x in coalRateDFL_l.LineagesPair]
coalRateDFL_l.loc[:, "Lineage2"] = [x[1] for x in coalRateDFL_l.LineagesPair]
coalRateDFL_l.loc[:, "Colour1"] = [lineages_colours()[x] for x in coalRateDFL_l.Lineage1]
coalRateDFL_l.loc[:, "Colour2"] = [lineages_colours()[x] for x in coalRateDFL_l.Lineage2]




/home/jobste/bin/anaconda3/envs/tsinfer/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/jobste/bin/anaconda3/envs/tsinfer/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [81]:
coalRates_flatten_s = []
for pair in range(len(subspeciesPairs)):
    coalRates_flatten_s.append(coal_rate_s[pair, :, :].flatten())

coalRateDF_s = pd.DataFrame.from_records(coalRates_flatten_s).T
coalRateDF_s.columns = subspeciesPairs
coalRateDF_s.loc[:, "Time"] = timeBreaks[1:]
coalRateDFL_s = pd.melt(coalRateDF_s, id_vars = "Time", var_name = "SubspeciesPair", value_name="CoalRate")


coalRateDFL_s.loc[:, "PairType"] = ["Within" if x[0] == x[1] else "Cross" for x in coalRateDFL_s.SubspeciesPair]
coalRateDFL_s.loc[:, "Ne"] = 1 / coalRateDFL_s.CoalRate
coalRateDFL_s.loc[:, "LogNe"] = np.log10(coalRateDFL_s.Ne)
coalRateDFL_s.loc[:, "LogTime"] = np.log10(coalRateDFL_s.Time)
coalRateDFL_s.loc[:, "Subspecies1"] = [x[0] for x in coalRateDFL_s.SubspeciesPair]
coalRateDFL_s.loc[:, "Subspecies2"] = [x[1] for x in coalRateDFL_s.SubspeciesPair]
coalRateDFL_s.loc[:, "Colour1"] = [subspecies_colours()[x] for x in coalRateDFL_s.Subspecies1]
coalRateDFL_s.loc[:, "Colour2"] = [subspecies_colours()[x] for x in coalRateDFL_s.Subspecies2]


/home/jobste/bin/anaconda3/envs/tsinfer/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [87]:
#LINEAGE
ggplot(data = coalRateDFL_l[coalRateDFL_l.PairType == "Within"]) + geom_step(aes(x = "LogTime", y = "LogNe", color = "Lineage1")) 
#+ \
#scale_color_manual(values = sorted(set(coalRateDFL_l.Colour1), key=coalRateDFL_l.Colour1.to_list().index))

In [88]:
name = "A"
plotDF =  pd.concat([coalRateDFL_l[(coalRateDFL_l.PairType == "Cross") & ((coalRateDFL_l.Lineage1 == name) | (coalRateDFL_l.Lineage2 == name))], coalRateDFL_l[(coalRateDFL_l.PairType == "Within") & (coalRateDFL_l.Lineage1 == name)]])
otherCol = get_lineage_colour([x if x != name else y for x,y in zip(plotDF.Lineage1, plotDF.Lineage2) ])
ggplot(data = plotDF) + geom_step(aes(x = "LogTime", y = "LogCoalRate", color = "LineagesPair")) 
#  scale_color_manual(values = sorted(set(otherCol), key=otherCol.index))

In [90]:
name = "M"
plotDF =  pd.concat([coalRateDFL_l[(coalRateDFL_l.PairType == "Cross") & ((coalRateDFL_l.Lineage1 == name) | (coalRateDFL_l.Lineage2 == name))], coalRateDFL_l[(coalRateDFL_l.PairType == "Within") & (coalRateDFL_l.Lineage1 == name)]])
otherCol = get_lineage_colour([x if x != name else y for x,y in zip(plotDF.Lineage1, plotDF.Lineage2) ])
ggplot(data = plotDF) + geom_step(aes(x = "LogTime", y = "CoalRate", color = "LineagesPair")) 

In [92]:
name = "C"
plotDF =  pd.concat([coalRateDFL_l[(coalRateDFL_l.PairType == "Cross") & ((coalRateDFL_l.Lineage1 == name) | (coalRateDFL_l.Lineage2 == name))], coalRateDFL_l[(coalRateDFL_l.PairType == "Within") & (coalRateDFL_l.Lineage1 == name)]])
otherCol = get_lineage_colour([x if x != name else y for x,y in zip(plotDF.Lineage1, plotDF.Lineage2) ])
ggplot(data = plotDF) + geom_step(aes(x = "LogTime", y = "CoalRate", color = "LineagesPair")) 

In [93]:
name = "O"
plotDF =  pd.concat([coalRateDFL_l[(coalRateDFL_l.PairType == "Cross") & ((coalRateDFL_l.Lineage1 == name) | (coalRateDFL_l.Lineage2 == name))], coalRateDFL_l[(coalRateDFL_l.PairType == "Within") & (coalRateDFL_l.Lineage1 == name)]])
otherCol = get_lineage_colour([x if x != name else y for x,y in zip(plotDF.Lineage1, plotDF.Lineage2) ])
ggplot(data = plotDF) + geom_step(aes(x = "LogTime", y = "CoalRate", color = "LineagesPair")) 

In [95]:
#SUBSPECIES
ggplot(data = coalRateDFL_s[(coalRateDFL_s.PairType == "Within") & (coalRateDFL_s.LogTime < 5)]) + geom_step(aes(x = "LogTime", y = "LogNe", color = "Subspecies1")) + \
scale_color_manual(values = sorted(set(coalRateDFL_s.Colour1), key=coalRateDFL_s.Colour1.to_list().index))

In [97]:
name = "Scutellata"
plotDF =  pd.concat([coalRateDFL_s[(coalRateDFL_s.PairType == "Cross") & ((coalRateDFL_s.Subspecies1 == name) | (coalRateDFL_s.Subspecies2 == name))], coalRateDFL_s[(coalRateDFL_s.PairType == "Within") & (coalRateDFL_s.Subspecies1 == name)]])
otherSubspCol = get_subsp_colour([x if x != name else y for x,y in zip(plotDF.Subspecies1, plotDF.Subspecies2) ])
ggplot(data = plotDF) + geom_step(aes(x = "LogTime", y = "CoalRate", color = "SubspeciesPair")) + \
  scale_color_manual(values = sorted(set(otherSubspCol), key=otherSubspCol.index))

In [18]:
name = "Capensis"
plotDF =  pd.concat([coalRateDFL_s[(coalRateDFL_s.PairType == "Cross") & ((coalRateDFL_s.Subspecies1 == name) | (coalRateDFL_s.Subspecies2 == name))], coalRateDFL_s[(coalRateDFL_s.PairType == "Within") & (coalRateDFL_s.Subspecies1 == name)]])
otherSubspCol = get_subsp_colour([x if x != name else y for x,y in zip(plotDF.Subspecies1, plotDF.Subspecies2) ])
ggplot(data = plotDF) + geom_line(aes(x = "LogTime", y = "CoalRate", color = "SubspeciesPair")) + \
  scale_color_manual(values = sorted(set(otherSubspCol), key=otherSubspCol.index)) 

In [22]:
name = "Unicolor"
plotDF =  pd.concat([coalRateDFL_s[(coalRateDFL_s.PairType == "Cross") & ((coalRateDFL_s.Subspecies1 == name) | (coalRateDFL_s.Subspecies2 == name))], coalRateDFL_s[(coalRateDFL_s.PairType == "Within") & (coalRateDFL_s.Subspecies1 == name)]])
otherSubspCol = get_subsp_colour([x if x != name else y for x,y in zip(plotDF.Subspecies1, plotDF.Subspecies2) ])
ggplot(data = plotDF) + geom_line(aes(x = "LogTime", y = "CoalRate", color = "SubspeciesPair")) + \
  scale_color_manual(values = sorted(set(otherSubspCol), key=otherSubspCol.index))

In [26]:
name = "Mellifera"
plotDF =  pd.concat([coalRateDFL_s[(coalRateDFL_s.PairType == "Cross") & ((coalRateDFL_s.Subspecies1 == name) | (coalRateDFL_s.Subspecies2 == name))], coalRateDFL_s[(coalRateDFL_s.PairType == "Within") & (coalRateDFL_s.Subspecies1 == name)]])
otherSubspCol = get_subsp_colour([x if x != name else y for x,y in zip(plotDF.Subspecies1, plotDF.Subspecies2) ])
ggplot(data = plotDF) + geom_line(aes(x = "LogTime", y = "CoalRate", color = "SubspeciesPair")) + \
  scale_color_manual(values = sorted(set(otherSubspCol), key=otherSubspCol.index))

In [27]:
name = "Iberiensis"
plotDF =  coalRateDFL_s[(coalRateDFL_s.PairType == "Cross") & ((coalRateDFL_s.Subspecies1 == name) | (coalRateDFL_s.Subspecies2 == name))]
otherSubspCol = get_subsp_colour([x if x != name else y for x,y in zip(plotDF.Subspecies1, plotDF.Subspecies2) ])
ggplot(data = plotDF) + geom_line(aes(x = "LogTime", y = "CoalRate", color = "SubspeciesPair")) + \
  scale_color_manual(values = sorted(set(otherSubspCol), key=otherSubspCol.index))

In [28]:
name = "Ligustica"
plotDF =  coalRateDFL_s[(coalRateDFL_s.PairType == "Cross") & ((coalRateDFL_s.Subspecies1 == name) | (coalRateDFL_s.Subspecies2 == name))]
otherSubspCol = get_subsp_colour([x if x != name else y for x,y in zip(plotDF.Subspecies1, plotDF.Subspecies2) ])
ggplot(data = plotDF) + geom_line(aes(x = "LogTime", y = "CoalRate", color = "SubspeciesPair")) + \
  scale_color_manual(values = sorted(set(otherSubspCol), key=otherSubspCol.index))

In [32]:
name = "Carnica"
plotDF =  pd.concat([coalRateDFL_s[(coalRateDFL_s.PairType == "Cross") & ((coalRateDFL_s.Subspecies1 == name) | (coalRateDFL_s.Subspecies2 == name))], coalRateDFL_s[(coalRateDFL_s.PairType == "Within") & (coalRateDFL_s.Subspecies1 == name)]])
otherSubspCol = get_subsp_colour([x if x != name else y for x,y in zip(plotDF.Subspecies1, plotDF.Subspecies2) ])
ggplot(data = plotDF) + geom_line(aes(x = "LogTime", y = "CoalRate", color = "SubspeciesPair")) + \
  scale_color_manual(values = sorted(set(otherSubspCol), key=otherSubspCol.index))

In [31]:
name = "Unicolor"
plotDF =  pd.concat([coalRateDFL_s[(coalRateDFL_s.PairType == "Cross") & ((coalRateDFL_s.Subspecies1 == name) | (coalRateDFL_s.Subspecies2 == name))], coalRateDFL_s[(coalRateDFL_s.PairType == "Within") & (coalRateDFL_s.Subspecies1 == name)]])
otherSubspCol = get_subsp_colour([x if x != name else y for x,y in zip(plotDF.Subspecies1, plotDF.Subspecies2) ])
ggplot(data = plotDF) + geom_line(aes(x = "LogTime", y = "CoalRate", color = "SubspeciesPair")) + \
  scale_color_manual(values = sorted(set(otherSubspCol), key=otherSubspCol.index))

In [33]:
name = "Buckfast"
plotDF =  pd.concat([coalRateDFL_s[(coalRateDFL_s.PairType == "Cross") & ((coalRateDFL_s.Subspecies1 == name) | (coalRateDFL_s.Subspecies2 == name))], coalRateDFL_s[(coalRateDFL_s.PairType == "Within") & (coalRateDFL_s.Subspecies1 == name)]])
otherSubspCol = get_subsp_colour([x if x != name else y for x,y in zip(plotDF.Subspecies1, plotDF.Subspecies2) ])
ggplot(data = plotDF) + geom_line(aes(x = "LogTime", y = "CoalRate", color = "SubspeciesPair")) + \
  scale_color_manual(values = sorted(set(otherSubspCol), key=otherSubspCol.index))